In [1]:
import cv2
from ultralytics import YOLO
import time
from datetime import datetime
from collections import deque

# Stability buffer
history = deque(maxlen=5)

# Light state memory
last_light_state = None
last_change_time = time.time()

# Load YOLO model
model = YOLO("yolov8n.pt")

# Start camera
cap = cv2.VideoCapture(0)

cap.set(3,640)
cap.set(4,480)

# Device power ratings
light_power = 40


total_energy_saved = 0

def get_stable_count(new_count):
    history.append(new_count)
    return sum(history) // len(history)


def get_light_level(count):
    if count == 0:
        return "OFF"
    elif 1 <= count <= 5:
        return "LOW"
    elif 6 <= count <= 20:
        return "MEDIUM"
    else:
        return "FULL"


def stable_light_control(count):
    global last_light_state, last_change_time

    new_state = get_light_level(count)

    # Only change if stable for 3 seconds
    if new_state != last_light_state:
        if time.time() - last_change_time > 3:
            last_light_state = new_state
            last_change_time = time.time()
    else:
        last_change_time = time.time()

    return last_light_state

while True:

    start_time = time.time()

    ret, frame = cap.read()

    if not ret:
        print("Camera not detected")
        break

    results = model(frame, verbose=False)

    person_count = 0

    for box in results[0].boxes:
        cls = int(box.cls)

        if cls == 0:
            person_count += 1
            
    # Apply smoothing
    stable_count = get_stable_count(person_count)

    # Get stable light level
    light_level = stable_light_control(stable_count)
    


    # DEVICE CONTROL LOGIC

    if light_level == "OFF":
    lights = "OFF"
    energy_usage = 0
    total_energy_saved += light_power

    elif light_level == "LOW":
    lights = "DIM"
    energy_usage = light_power * 0.5

    elif light_level == "MEDIUM":
    lights = "HALF"
    energy_usage = light_power * 0.75

    elif light_level == "FULL":
    lights = "FULL"
    energy_usage = light_power

    # FPS calculation
    end_time = time.time()
    fps = 1/(end_time-start_time)

    annotated_frame = results[0].plot()

    now = datetime.now()
    current_time = now.strftime("%Y-%m-%d %H:%M:%S")

    # Display information
    
    cv2.putText(annotated_frame,current_time,(20,320),
                cv2.FONT_HERSHEY_SIMPLEX,0.7,(200,200,200),2)

    cv2.putText(annotated_frame,f"Occupancy: {stable_count}",(20,40),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,0),2)

    cv2.putText(annotated_frame,f"Lights: {lights}",(20,80),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,255,255),2)


    cv2.putText(annotated_frame,f"Energy Usage: {energy_usage} W",(20,200),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,0),2)

    cv2.putText(annotated_frame,f"Energy Saved: {total_energy_saved} W",(20,240),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(0,200,255),2)

    cv2.putText(annotated_frame,f"FPS: {int(fps)}",(20,280),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,0,0),2)


    cv2.imshow("Smart Classroom Energy Monitoring",annotated_frame)

    key = cv2.waitKey(1)

    if key == ord('q') or key == 27:
        break

cap.release()
cv2.destroyAllWindows()

IndentationError: expected an indented block after 'if' statement on line 90 (4087625390.py, line 91)